In [0]:
from pyspark.sql.functions import col, trim, initcap, to_date

bronze_sales = spark.table("workspace.freshmart_bronze.raw_sales")

# Standardize text and convert numbers/dates safely
standardized = (
    bronze_sales
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("product_name", initcap(trim(col("product_name"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("qty", col("qty").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .dropDuplicates(["order_id"])
)

# Data Quality Rule: valid orders must have positive price, positive qty, and a product name
valid_condition = (
    (col("unit_price") > 0) &
    (col("qty") > 0) &
    col("product_name").isNotNull() &
    (col("product_name") != "")
)

clean_sales = standardized.filter(valid_condition).withColumn(
    "total_amount", col("qty") * col("unit_price")
)
bad_sales = standardized.filter(~valid_condition)

# Write Clean Silver Table & Quarantine Table
clean_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.freshmart_silver.clean_sales")
bad_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.freshmart_silver.quarantine_sales")

print(f"Clean rows: {clean_sales.count()} | Quarantined bad rows: {bad_sales.count()}")

Clean rows: 4 | Quarantined bad rows: 2


In [0]:
%sql
-- 1. Create the Silver SCD Type 2 Customer table if it does not exist yet
CREATE TABLE IF NOT EXISTS workspace.freshmart_silver.scd_customers (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id STRING,
  customer_name STRING,
  city STRING,
  effective_from DATE,
  effective_to DATE,
  is_current BOOLEAN
) USING DELTA;

-- 2. Prepare latest cleaned incoming customer records
CREATE OR REPLACE TEMP VIEW incoming_customers AS
SELECT
  customer_id,
  INITCAP(TRIM(customer_name)) AS customer_name,
  INITCAP(TRIM(city)) AS city,
  TO_DATE(updated_at, 'yyyy-MM-dd') AS updated_at
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC, ingestion_time DESC) AS rn
  FROM workspace.freshmart_bronze.raw_customers
)
WHERE rn = 1;

-- 3. Step A of SCD Type 2: Close old active row if the customer changed city or name
MERGE INTO workspace.freshmart_silver.scd_customers AS target
USING incoming_customers AS source
ON target.customer_id = source.customer_id AND target.is_current = true
WHEN MATCHED AND (target.city <> source.city OR target.customer_name <> source.customer_name)
THEN UPDATE SET
  target.effective_to = source.updated_at,
  target.is_current = false;

-- 4. Step B of SCD Type 2: Insert brand-new customers OR the new active row for changed customers
INSERT INTO workspace.freshmart_silver.scd_customers (customer_id, customer_name, city, effective_from, effective_to, is_current)
SELECT
  source.customer_id,
  source.customer_name,
  source.city,
  source.updated_at AS effective_from,
  NULL AS effective_to,
  true AS is_current
FROM incoming_customers AS source
LEFT JOIN workspace.freshmart_silver.scd_customers AS target
  ON source.customer_id = target.customer_id AND target.is_current = true
WHERE target.customer_id IS NULL;

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM workspace.freshmart_silver.quarantine_sales

order_id,order_date,customer_id,product_id,product_name,category,qty,unit_price,_rescued_data,ingestion_time,source_file
O104,2026-11-02,C001,P02,Coffee Mug,Kitchen,1,-250.0,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
O105,2026-11-03,C002,P04,null,Electronics,1,1500.0,null,2026-11-25T17:24:01.062Z,/Volumes/workspace/freshmart_bronze/raw_files/sales/sales_day1.csv
